# Stochastic Gradient Descent (SGD)
"A Stochastic Approximation Method" (Robbins & Monro, 1951); momentum (Polyak, 1964); Nesterov momentum (Nesterov, 1983; Sutskever et al., 2013)

**Idea:** the loss is an average over the whole dataset, but its gradient can be **estimated** from a small random batch. The estimate is noisy but unbiased and $N / B$ times cheaper, so we take many cheap noisy steps instead of a few exact ones. Momentum then averages the noisy steps over time.

### GD vs SGD vs Momentum vs Adam
| | Gradient descent | SGD (mini-batch) | SGD + momentum | Adam / AdamW |
|---|---|---|---|---|
| Gradient from | all $N$ examples | $B$ random examples | $B$ random examples | $B$ random examples |
| Cost per step | $O(N)$ | $O(B)$ | $O(B)$ | $O(B)$ |
| Steps per epoch | 1 | $N / B$ | $N / B$ | $N / B$ |
| State per parameter | - | - | velocity $v$ | $m$ and $v$ (1st and 2nd moment) |
| Step size | one global $\eta$ | one global $\eta$ | one global $\eta$, effective $\eta / (1 - \mu)$ | **per-parameter**, $\eta / \sqrt{\hat v}$ |
| Used for | small convex problems | - | CNNs (ResNet, VGG: SGD + Nesterov) | Transformers (see `bert.ipynb`) |

Here: an optimizer written from scratch and checked against `torch.optim.SGD`, a 2D loss surface to see what momentum does, how noisy a mini-batch gradient really is, and batch size / momentum / schedule experiments on MNIST.

Update flow: `batch (B, 784) -> loss (scalar) -> grads (same shape as params) -> v = mu * v + g -> params -= lr * v`

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)

## 1. The update rules
All four are the same loop; they differ in what is subtracted from the parameters.

### Formula: Gradient descent (full batch)
$$\theta_{t+1} = \theta_t - \eta\, \nabla L(\theta_t), \qquad L(\theta) = \frac{1}{N}\sum_{i=1}^{N} \ell(x_i, y_i; \theta)$$
- $\theta$: all parameters; $\eta$: learning rate
- $\ell$: loss of one example; $N$: dataset size (60,000 for MNIST)
- one step costs a full pass over the data

### Formula: SGD (mini-batch)
$$\theta_{t+1} = \theta_t - \eta\, g_t, \qquad g_t = \frac{1}{B}\sum_{i \in \mathcal{B}_t} \nabla \ell(x_i, y_i; \theta_t)$$
- $\mathcal{B}_t$: a random batch of $B$ examples
- $\mathbb{E}[g_t] = \nabla L(\theta_t)$: **unbiased**; $\text{Var}[g_t] \propto 1 / B$
- "SGD" in the 1951 sense is $B = 1$; in deep learning it always means mini-batch

### Formula: Momentum (heavy ball)
$$v_{t+1} = \mu\, v_t + g_t, \qquad \theta_{t+1} = \theta_t - \eta\, v_{t+1}$$
- $v$: velocity, an exponentially weighted sum of past gradients, $v_0 = 0$
- $\mu$: momentum (0.9); a constant gradient $g$ builds up to $v = g / (1 - \mu)$, so the effective step is $10\times$ larger
- directions where the gradient keeps its sign accelerate; directions where it flips sign cancel out

### Formula: Nesterov momentum
$$v_{t+1} = \mu\, v_t + g_t, \qquad \theta_{t+1} = \theta_t - \eta\,(g_t + \mu\, v_{t+1})$$
- the step uses the velocity **one step ahead**: a "look-ahead" correction that brakes earlier when the gradient turns
- this is the PyTorch form (Sutskever et al., 2013), equivalent to evaluating the gradient at $\theta_t - \eta \mu v_t$

### Formula: Weight decay (L2)
$$g_t \leftarrow g_t + \lambda\, \theta_t$$
- $\lambda$: weight decay; for plain SGD this is exactly an L2 penalty $\frac{\lambda}{2}\lVert\theta\rVert^2$ added to the loss
- for Adam the two are *not* the same, which is why AdamW exists (see `bert.ipynb`)

In [ ]:
class SGD:
    # same update as torch.optim.SGD (dampening = 0)
    def __init__(self, params, lr, momentum=0.0, nesterov=False, weight_decay=0.0):
        self.params = list(params)
        self.lr, self.momentum, self.nesterov, self.weight_decay = lr, momentum, nesterov, weight_decay
        self.v = [torch.zeros_like(p) for p in self.params]   # one velocity per parameter tensor

    def zero_grad(self):
        for p in self.params:
            p.grad = None

    @torch.no_grad()
    def step(self):
        for p, v in zip(self.params, self.v):
            if p.grad is None:
                continue
            g = p.grad
            if self.weight_decay:
                g = g + self.weight_decay * p             # L2: g + lambda * theta
            if self.momentum:
                v.mul_(self.momentum).add_(g)             # v = mu * v + g
                g = g + self.momentum * v if self.nesterov else v
            p.sub_(self.lr * g)                           # theta = theta - lr * step

### Check against `torch.optim.SGD`
Two copies of the same small net, 20 steps each on the same batches. The parameters should match to float precision.

In [ ]:
def check(**kw):
    torch.manual_seed(0)
    a = nn.Sequential(nn.Linear(10, 16), nn.Tanh(), nn.Linear(16, 3))
    b = nn.Sequential(nn.Linear(10, 16), nn.Tanh(), nn.Linear(16, 3))
    b.load_state_dict(a.state_dict())
    ours, ref = SGD(a.parameters(), **kw), torch.optim.SGD(b.parameters(), **kw)
    for _ in range(20):
        x, y = torch.randn(32, 10), torch.randint(3, (32,))       # (B, 10), (B,)
        for net, opt in [(a, ours), (b, ref)]:
            opt.zero_grad()
            F.cross_entropy(net(x), y).backward()
            opt.step()
    return max((p - q).abs().max().item() for p, q in zip(a.parameters(), b.parameters()))

print("plain    :", check(lr=0.1))
print("momentum :", check(lr=0.1, momentum=0.9))
print("nesterov :", check(lr=0.1, momentum=0.9, nesterov=True, weight_decay=1e-2))

## 2. What momentum does: a narrow valley
An ill-conditioned quadratic: steep along $w_2$, flat along $w_1$. This is the shape of most loss surfaces locally (a few sharp directions, many flat ones).

### Formula: Toy loss
$$f(w) = \tfrac{1}{2}\left(w_1^2 + \kappa\, w_2^2\right), \qquad \nabla f(w) = (w_1,\ \kappa\, w_2)$$
- $\kappa = 100$: condition number (ratio of the largest to the smallest curvature)
- plain GD is stable only if $\eta < 2 / \kappa = 0.02$; with that small $\eta$ the flat direction shrinks by just $(1 - \eta)$ per step
- so GD either oscillates across the valley or crawls along it; momentum damps the first and speeds up the second

In [ ]:
kappa = 100.0
f = lambda w: 0.5 * (w[0] ** 2 + kappa * w[1] ** 2)

def trajectory(steps=100, **kw):
    w = torch.tensor([-9.0, 1.5], requires_grad=True)   # (2,)
    opt = SGD([w], **kw)
    path = [w.detach().clone()]
    for _ in range(steps):
        opt.zero_grad()
        f(w).backward()
        opt.step()
        path.append(w.detach().clone())
    return torch.stack(path)                             # (steps+1, 2)

runs = {
    "GD, lr 0.018": trajectory(lr=0.018),
    "momentum 0.9, lr 0.01": trajectory(lr=0.01, momentum=0.9),
    "Nesterov 0.9, lr 0.01": trajectory(lr=0.01, momentum=0.9, nesterov=True),
}

g1, g2 = torch.meshgrid(torch.linspace(-10, 4, 200), torch.linspace(-2.5, 2.5, 200), indexing="xy")
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].contour(g1, g2, 0.5 * (g1 ** 2 + kappa * g2 ** 2), levels=30, colors="lightgray", linewidths=0.7)
for name, p in runs.items():
    ax[0].plot(p[:, 0], p[:, 1], marker=".", ms=3, lw=1, label=name)
    ax[1].semilogy([f(w).item() for w in p], label=name)
    print(f"{name:24s} loss after 100 steps: {f(p[-1]).item():.2e}")
ax[0].set_xlabel("$w_1$ (flat)"); ax[0].set_ylabel("$w_2$ (steep)"); ax[0].legend()
ax[1].set_xlabel("step"); ax[1].set_ylabel("loss"); ax[1].legend()
plt.show()

## 3. Data - MNIST, held on the GPU
60k train / 10k test images of 28x28 digits, flattened to 784 and normalized. The whole dataset fits in memory, so batches are just index lookups (no `DataLoader`), which keeps the batch-size experiments fast.

The model is a 2-layer MLP, `784 -> 256 -> 10`.

In [ ]:
train_ds = datasets.MNIST("../data", train=True, download=True)
test_ds = datasets.MNIST("../data", train=False, download=True)

def prep(ds):
    x = (ds.data.float() / 255 - 0.1307) / 0.3081         # (N, 28, 28)
    return x.reshape(len(x), -1).to(device), ds.targets.to(device)  # (N, 784), (N,)

Xtr, ytr = prep(train_ds)
Xte, yte = prep(test_ds)
N = len(Xtr)

def make_model(seed=0):
    torch.manual_seed(seed)                                # same init for every run
    return nn.Sequential(nn.Linear(784, 256), nn.ReLU(), nn.Linear(256, 10)).to(device)

@torch.no_grad()
def test_acc(model):
    return (model(Xte).argmax(-1) == yte).float().mean().item()

print(Xtr.shape, ytr.shape, Xte.shape)

## 4. How noisy is a mini-batch gradient?
Compute the **full** gradient $\nabla L$ over all 60k examples once, then compare mini-batch gradients $g$ of different sizes against it.

### Formula: Gradient noise
$$g = \nabla L + \epsilon, \qquad \mathbb{E}[\epsilon] = 0, \qquad \mathbb{E}\,\lVert\epsilon\rVert^2 = \frac{\sigma^2}{B}$$
- $\sigma^2$: variance of the per-example gradients around their mean
- the noise **norm** falls as $1 / \sqrt{B}$: $100\times$ more compute buys only $10\times$ less noise
- cosine similarity $\cos(g, \nabla L)$ shows how well a batch points in the true downhill direction

In [ ]:
def flat_grad(model, x, y):
    # -> all gradients as one vector (P,)
    model.zero_grad()
    F.cross_entropy(model(x), y).backward()
    return torch.cat([p.grad.flatten() for p in model.parameters()])

model = make_model()
full = flat_grad(model, Xtr, ytr)                           # (P,) exact gradient
print(f"params: {full.numel()} | full-gradient norm: {full.norm().item():.4f}")
for B in [1, 8, 64, 512, 4096]:
    cos, err = [], []
    for _ in range(50):
        ix = torch.randint(N, (B,), device=device)
        g = flat_grad(model, Xtr[ix], ytr[ix])              # (P,) noisy estimate
        cos.append(F.cosine_similarity(g, full, dim=0).item())
        err.append((g - full).norm().item())
    print(f"B = {B:5d} | cos(g, full) {sum(cos) / 50:.3f} | noise norm {sum(err) / 50:.4f} | noise * sqrt(B) {sum(err) / 50 * B ** 0.5:.3f}")

## 5. Batch size: same compute, different number of updates
Every run below sees the data exactly `epochs` times, so the cost in example-gradients is identical. What changes is how many **updates** that budget buys: $N / B$ per epoch.

Full-batch GD gets 3 exact steps; $B = 16$ gets 11,250 noisy ones.

In [ ]:
def train(opt_fn, B=128, epochs=3, schedule=None, log_every=50, seed=0):
    # -> model, [(examples seen, train loss)]
    model = make_model(seed)
    opt = opt_fn(model.parameters())
    base_lr = opt.lr
    total = epochs * math.ceil(N / B)
    g = torch.Generator(device=device).manual_seed(seed)
    curve, step, run = [], 0, 0.0
    for epoch in range(epochs):
        perm = torch.randperm(N, device=device, generator=g)   # reshuffle every epoch
        for i in range(0, N, B):
            ix = perm[i:i + B]
            if schedule:
                opt.lr = base_lr * schedule(step, total)
            loss = F.cross_entropy(model(Xtr[ix]), ytr[ix])    # (B, 10) vs (B,) -> scalar
            opt.zero_grad()
            loss.backward()
            opt.step()
            step += 1; run += loss.item()
            if step % log_every == 0 or step == total:
                n = log_every if step % log_every == 0 else step % log_every
                curve.append((min(step * B, epochs * N), run / n)); run = 0.0
    return model, curve

for B in [16, 128, 1024, N]:
    model, curve = train(lambda p: SGD(p, lr=0.1), B=B, log_every=max(1, 3000 // B))
    plt.plot(*zip(*curve), label=f"B = {B}")
    print(f"B = {B:5d} | updates {3 * math.ceil(N / B):6d} | final train loss {curve[-1][1]:.4f} | test acc {test_acc(model):.4f}")
plt.xlabel("examples seen"); plt.ylabel("train loss"); plt.yscale("log"); plt.legend(); plt.show()

## 6. Momentum and Nesterov on MNIST
Same model, same batches ($B = 128$), 3 epochs. Momentum 0.9 multiplies the effective step by $1 / (1 - \mu) = 10$, so plain SGD is shown at both the same $\eta$ and at $10\eta$ - the fair comparison is against the second.

In [ ]:
configs = {
    "SGD, lr 0.01": lambda p: SGD(p, lr=0.01),
    "SGD, lr 0.1": lambda p: SGD(p, lr=0.1),
    "momentum 0.9, lr 0.01": lambda p: SGD(p, lr=0.01, momentum=0.9),
    "Nesterov 0.9, lr 0.01": lambda p: SGD(p, lr=0.01, momentum=0.9, nesterov=True),
}
for name, opt_fn in configs.items():
    model, curve = train(opt_fn)
    plt.plot(*zip(*curve), label=name, lw=1)
    print(f"{name:24s} | final train loss {curve[-1][1]:.4f} | test acc {test_acc(model):.4f}")
plt.xlabel("examples seen"); plt.ylabel("train loss"); plt.yscale("log"); plt.legend(); plt.show()

## 7. Learning-rate schedule
The run below uses a deliberately large $\eta = 0.1$ with momentum 0.9 for 8 epochs. With a constant $\eta$, SGD never settles: the gradient noise keeps kicking the parameters around the minimum, in a ball whose radius grows with $\eta$. Shrinking $\eta$ over time shrinks the ball.

### Formula: Robbins-Monro conditions
$$\sum_{t=1}^{\infty} \eta_t = \infty, \qquad \sum_{t=1}^{\infty} \eta_t^2 < \infty$$
- first: the steps add up to infinity, so any starting point can reach the minimum
- second: the steps shrink fast enough for the accumulated noise to stay finite
- e.g. $\eta_t = \eta_0 / t$ satisfies both; a constant $\eta$ fails the second

### Formula: Cosine decay (what is used in practice)
$$\eta_t = \frac{\eta}{2}\left(1 + \cos\frac{\pi t}{T_{total}}\right)$$
- $\eta$: peak learning rate; $t$: step; $T_{total}$: total steps
- stays high early (fast progress), goes smoothly to 0 at the end (noise averages out)
- ResNet's original recipe is the cruder **step decay**: divide $\eta$ by 10 at fixed epochs

In [ ]:
cosine = lambda t, total: 0.5 * (1 + math.cos(math.pi * t / total))

for name, sched in [("constant", None), ("cosine", cosine)]:
    model, curve = train(lambda p: SGD(p, lr=0.1, momentum=0.9, nesterov=True, weight_decay=5e-4), epochs=8, schedule=sched)
    plt.plot(*zip(*curve), label=name, lw=1)
    with torch.no_grad():
        full_loss = F.cross_entropy(model(Xtr), ytr).item()    # loss over all 60k, not one noisy batch
    print(f"{name:9s} | full train loss {full_loss:.4f} | test acc {test_acc(model):.4f}")
plt.xlabel("examples seen"); plt.ylabel("train loss"); plt.yscale("log"); plt.legend(); plt.show()

## Notes
- The mini-batch gradient is a poor estimate one step at a time (see the cosine similarities in §4) and it does not matter: the errors are zero-mean, so they average out over steps. That is the whole reason deep learning is affordable.
- At equal compute, smaller batches win until the per-step overhead dominates; on a GPU the batch is usually set by memory and throughput, and $\eta$ is scaled with it (**linear scaling rule**: $B \times k \Rightarrow \eta \times k$, with warmup; Goyal et al., 2017).
- Gradient noise is also a regularizer: small-batch SGD tends to end in flatter minima that generalize better than large-batch training at the same loss (Keskar et al., 2017).
- Momentum mostly helps through the $1 / (1 - \mu)$ larger effective step while staying stable in the steep directions. On the toy valley Nesterov overshoots less than heavy ball; on MNIST the two are nearly identical, and both match plain SGD at $10\eta$.
- SGD + momentum + step/cosine decay is still the standard for CNNs (`convolution.ipynb`, VGG/ResNet). Transformers train poorly with one global step size and use Adam/AdamW (`slm.ipynb`, `bert.ipynb`), which divides each parameter's step by a running RMS of its gradient.
- Not covered: Adagrad, RMSProp, Adam (adaptive per-parameter rates), and second-order methods (L-BFGS is used in `style_transfer.ipynb`).